In [1]:

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
mes = 'mar_2026'

In [3]:
#Archivos 

cerrado = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/IRI_cerrrado.csv", delimiter=';')

etapa_1 = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/IRI_Etapa1.csv", delimiter=';')

turno_ccz = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Turnos_CCZ.csv",delimiter=';')

turno_via = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Turnos_Via.csv",delimiter=';')

turno_sup = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Turnos_Sup.csv",delimiter=';')

asistencia = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Asistencia.csv",encoding='latin', delimiter=';')

tipo_dia = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Tipo día.csv", encoding='latin',delimiter=';')

personal = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Personal_id.csv",encoding='latin',delimiter=';')

In [4]:
#consolidar iri con los datos de cerrado y etapa_1

iri = pd.concat([cerrado,etapa_1], ignore_index=True)

iri

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_7000\2545014935.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  iri = pd.concat([cerrado,etapa_1], ignore_index=True)


,IdIRI,Estado,Fecha Viaje,F. Inicio DP,F. Cierre DP,Fuente,Servicio,IdViaje,ViajeLinea,Coche,...,Imputación de datos,Primer Viaje,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia
0,86273059,No Contestado,1/03/2026,3/03/2026 0:00,9/03/2026 23:59,URBANO,AD0691017,1,1,17,...,Con hora de Inicio LG SMART OPERATOR,0,0,0.00,0.00,0,0,0,0,NaN
1,86280906,No Contestado,1/03/2026,3/03/2026 0:00,9/03/2026 23:59,ALIMENTACION,CE0A60012,28,27,2,...,Sin hora de Inicio - No Imputado,0,0,0.00,0.00,0,0,0,1,NaN
2,86280432,No Contestado,1/03/2026,3/03/2026 0:00,9/03/2026 23:59,URBANO,CE0650001,2,1,1,...,Con hora de Inicio LG SMART OPERATOR,1,0,0.00,0.02,200.000.000,1,0,1,NaN
3,86280433,No Contestado,1/03/2026,3/03/2026 0:00,9/03/2026 23:59,URBANO,CE0650001,3,2,1,...,Con hora de Inicio LG SMART OPERATOR,0,0,20.97,-0.48,200.000.000,1,0,1,NaN
4,86280434,No Contestado,1/03/2026,3/03/2026 0:00,9/03/2026 23:59,URBANO,CE0650001,4,3,1,...,Con hora de Inicio LG SMART OPERATOR,0,0,21.93,0.47,200.000.000,1,0,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,87398573,No Contestado,31/03/2026,6/04/2026 0:00,10/04/2026 23:59,ALIMENTACION,CO11B0012,21,20,6,...,Con hora de Inicio LG SMART OPERATOR,0,0,4.50,-0.97,54.545.455,1,0,1,NaN
113753,87398574,No Contestado,31/03/2026,6/04/2026 0:00,10/04/2026 23:59,ALIMENTACION,CO11B0012,22,21,6,...,Con hora de Inicio LG SMART OPERATOR,0,0,4.20,-1.92,54.545.455,1,0,1,NaN
113754,87398575,No Contestado,31/03/2026,6/04/2026 0:00,10/04/2026 23:59,ALIMENTACION,CO11B0012,23,22,6,...,Con hora de Inicio LG SMART OPERATOR,0,0,5.28,-2.47,66.666.667,1,0,1,NaN
113755,87398576,No Contestado,31/03/2026,6/04/2026 0:00,10/04/2026 23:59,ALIMENTACION,CO11B0012,24,23,6,...,Con hora de Inicio LG SMART OPERATOR,0,0,5.03,-2.12,66.666.667,1,0,1,NaN


In [5]:
#Eliminar columnas 

col_eliminar = ['IdIRI', 'Estado', 'F. Inicio DP', 'F. Cierre DP','Operador Programado','Id Operador','Operador']

iri = iri.drop(columns=col_eliminar)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Imputación de datos,Primer Viaje,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,Con hora de Inicio LG SMART OPERATOR,0,0,0.00,0.00,0,0,0,0,NaN
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,Sin hora de Inicio - No Imputado,0,0,0.00,0.00,0,0,0,1,NaN
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,Con hora de Inicio LG SMART OPERATOR,1,0,0.00,0.02,200.000.000,1,0,1,NaN
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,Con hora de Inicio LG SMART OPERATOR,0,0,20.97,-0.48,200.000.000,1,0,1,NaN
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,Con hora de Inicio LG SMART OPERATOR,0,0,21.93,0.47,200.000.000,1,0,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,Con hora de Inicio LG SMART OPERATOR,0,0,4.50,-0.97,54.545.455,1,0,1,NaN
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,Con hora de Inicio LG SMART OPERATOR,0,0,4.20,-1.92,54.545.455,1,0,1,NaN
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,Con hora de Inicio LG SMART OPERATOR,0,0,5.28,-2.47,66.666.667,1,0,1,NaN
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,Con hora de Inicio LG SMART OPERATOR,0,0,5.03,-2.12,66.666.667,1,0,1,NaN


In [6]:
iri['Cantidad'] = 1

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Primer Viaje,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0,0,0.00,0.00,0,0,0,0,NaN,1
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0,0,0.00,0.00,0,0,0,1,NaN,1
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,1,0,0.00,0.02,200.000.000,1,0,1,NaN,1
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,0,0,20.97,-0.48,200.000.000,1,0,1,NaN,1
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,0,0,21.93,0.47,200.000.000,1,0,1,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,0,0,4.50,-0.97,54.545.455,1,0,1,NaN,1
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,0,0,4.20,-1.92,54.545.455,1,0,1,NaN,1
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,0,0,5.28,-2.47,66.666.667,1,0,1,NaN,1
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,0,0,5.03,-2.12,66.666.667,1,0,1,NaN,1


In [7]:
#Convertir datos de columna Hora Teórica

# Expresión regular para extraer la hora
patron_hora = r'(\d{1,2}:\d{2}:\d{2}\s[ap]\.?\s?[mM]\.)'

# Aplicar la extracción de hora a la columna 'Hora Teórica'
iri['Hora'] = iri['Hora Teórica'].str.extract(patron_hora)

# Mostrar el DataFrame resultante
iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0,0.00,0.00,0,0,0,0,NaN,1,NaN
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0,0.00,0.00,0,0,0,1,NaN,1,NaN
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,0,0.00,0.02,200.000.000,1,0,1,NaN,1,NaN
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,0,20.97,-0.48,200.000.000,1,0,1,NaN,1,NaN
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,0,21.93,0.47,200.000.000,1,0,1,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,0,4.50,-0.97,54.545.455,1,0,1,NaN,1,NaN
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,0,4.20,-1.92,54.545.455,1,0,1,NaN,1,NaN
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,0,5.28,-2.47,66.666.667,1,0,1,NaN,1,NaN
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,0,5.03,-2.12,66.666.667,1,0,1,NaN,1,NaN


In [8]:
# Dividir la columna 'Hora Teórica' en 'Fecha' y 'Hora'
iri[['Fecha1', 'Hora']] = iri['Hora Teórica'].str.split(' ', n=1, expand=True)

# Mostrar el DataFrame resultante
iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0.00,0.00,0,0,0,0,NaN,1,NaN,NaN
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0.00,0.00,0,0,0,1,NaN,1,16:02,1/03/2026
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,0.00,0.02,200.000.000,1,0,1,NaN,1,5:00,1/03/2026
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,20.97,-0.48,200.000.000,1,0,1,NaN,1,5:44,1/03/2026
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,21.93,0.47,200.000.000,1,0,1,NaN,1,6:28,1/03/2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,4.50,-0.97,54.545.455,1,0,1,NaN,1,17:14,31/03/2026
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,4.20,-1.92,54.545.455,1,0,1,NaN,1,17:55,31/03/2026
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,5.28,-2.47,66.666.667,1,0,1,NaN,1,18:38,31/03/2026
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,5.03,-2.12,66.666.667,1,0,1,NaN,1,19:22,31/03/2026


In [9]:
# Dividir la columna  en partes usando ':', y seleccionar la primera parte (horas)
iri['franja'] = iri['Hora'].str.split(':').str[0]

# Convertir la columna 'franja' a tipo entero
iri['franja'] = pd.to_numeric(iri['franja'], errors='coerce').astype('Int64')

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0.00,0,0,0,0,NaN,1,NaN,NaN,<NA>
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0.00,0,0,0,1,NaN,1,16:02,1/03/2026,16
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,0.02,200.000.000,1,0,1,NaN,1,5:00,1/03/2026,5
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,-0.48,200.000.000,1,0,1,NaN,1,5:44,1/03/2026,5
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,0.47,200.000.000,1,0,1,NaN,1,6:28,1/03/2026,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,-0.97,54.545.455,1,0,1,NaN,1,17:14,31/03/2026,17
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,-1.92,54.545.455,1,0,1,NaN,1,17:55,31/03/2026,17
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,-2.47,66.666.667,1,0,1,NaN,1,18:38,31/03/2026,18
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,-2.12,66.666.667,1,0,1,NaN,1,19:22,31/03/2026,19


In [10]:
#Llevar ruta comercial a iri

def calcular_posicion(Linea):
    
    filtro = (
        (turno_via['Linea'] == Linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_via.loc[filtro].empty:
        # Obtener el primer valor
        ruta = turno_via.loc[filtro, 'Ruta'].iloc[0]
        return ruta  if not pd.isna(ruta) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Ruta_Comercial'] = iri.apply(
    lambda row: calcular_posicion(
        row['Linea SAE']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0,0,0,0,NaN,1,NaN,NaN,<NA>,None
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0,0,0,1,NaN,1,16:02,1/03/2026,16,16-14
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,200.000.000,1,0,1,NaN,1,5:00,1/03/2026,5,None
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,200.000.000,1,0,1,NaN,1,5:44,1/03/2026,5,None
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,200.000.000,1,0,1,NaN,1,6:28,1/03/2026,6,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,54.545.455,1,0,1,NaN,1,17:14,31/03/2026,17,1-sep
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,54.545.455,1,0,1,NaN,1,17:55,31/03/2026,17,1-sep
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,66.666.667,1,0,1,NaN,1,18:38,31/03/2026,18,1-sep
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,66.666.667,1,0,1,NaN,1,19:22,31/03/2026,19,1-sep


In [11]:
#Llevar tipo día a iri

def calcular_posicion(fecha):
    
    filtro = (
        (tipo_dia['Fecha'] == fecha)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not tipo_dia.loc[filtro].empty:
        # Obtener el primer valor
        Tipo_día = tipo_dia.loc[filtro, 'Tipo Día'].iloc[0]
        return Tipo_día  if not pd.isna(Tipo_día ) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Tipo dia'] = iri.apply(
    lambda row: calcular_posicion(
        row['Fecha Viaje']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0,0,0,NaN,1,NaN,NaN,<NA>,None,Festivo
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0,0,1,NaN,1,16:02,1/03/2026,16,16-14,Festivo
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,1,0,1,NaN,1,5:00,1/03/2026,5,None,Festivo
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,1,0,1,NaN,1,5:44,1/03/2026,5,None,Festivo
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,1,0,1,NaN,1,6:28,1/03/2026,6,None,Festivo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,1,0,1,NaN,1,17:14,31/03/2026,17,1-sep,Hábil
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,1,0,1,NaN,1,17:55,31/03/2026,17,1-sep,Hábil
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,1,0,1,NaN,1,18:38,31/03/2026,18,1-sep,Hábil
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,1,0,1,NaN,1,19:22,31/03/2026,19,1-sep,Hábil


In [12]:
# Función para obtener la letra inicial de cada palabra
def obtener_letra_inicial(texto):
    palabras = texto.split()
    iniciales = [palabra[0] for palabra in palabras]
    return ''.join(iniciales)

# Aplicar la función a la columna 'Tipo día'
iri['Tipo_dia'] = iri['Tipo dia'].apply(obtener_letra_inicial)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0,0,NaN,1,NaN,NaN,<NA>,None,Festivo,F
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,0,1,NaN,1,16:02,1/03/2026,16,16-14,Festivo,F
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,0,1,NaN,1,5:00,1/03/2026,5,None,Festivo,F
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,0,1,NaN,1,5:44,1/03/2026,5,None,Festivo,F
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,0,1,NaN,1,6:28,1/03/2026,6,None,Festivo,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,0,1,NaN,1,17:14,31/03/2026,17,1-sep,Hábil,H
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,0,1,NaN,1,17:55,31/03/2026,17,1-sep,Hábil,H
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,0,1,NaN,1,18:38,31/03/2026,18,1-sep,Hábil,H
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,0,1,NaN,1,19:22,31/03/2026,19,1-sep,Hábil,H


In [13]:
#Traer el turno de reg_via

def calcular_turno(Linea,Ruta_SAE,Franja,Tipo_Dia):
    
    filtro = (
        (turno_via['Linea'] == Linea)&
        (turno_via['Ruta_SAE'] == Ruta_SAE)&
        (turno_via['Franja'] == Franja)&
        (turno_via['Tipo Dia'] == Tipo_Dia)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_via.loc[filtro].empty:
        # Obtener el primer valor
        turno = turno_via.loc[filtro, 'Turno'].iloc[0]
        return turno if not pd.isna(turno) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Turno_via'] = iri.apply(
    lambda row: calcular_turno(
        row['Linea SAE'],
        row['Ruta SAE'],
        row['franja'],
        row['Tipo_dia']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,0,NaN,1,NaN,NaN,<NA>,None,Festivo,F,None
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,1,NaN,1,16:02,1/03/2026,16,16-14,Festivo,F,PLA2
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,1,NaN,1,5:00,1/03/2026,5,None,Festivo,F,None
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,1,NaN,1,5:44,1/03/2026,5,None,Festivo,F,None
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,1,NaN,1,6:28,1/03/2026,6,None,Festivo,F,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,1,NaN,1,17:14,31/03/2026,17,1-sep,Hábil,H,None
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,1,NaN,1,17:55,31/03/2026,17,1-sep,Hábil,H,None
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,1,NaN,1,18:38,31/03/2026,18,1-sep,Hábil,H,None
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,1,NaN,1,19:22,31/03/2026,19,1-sep,Hábil,H,None


In [14]:
#Traer el turno de control

def calcular_turno(Linea,Ruta_SAE,Franja,Tipo_Dia):
    
    filtro = (
        (turno_ccz['Linea'] == Linea)&
        (turno_ccz['Ruta_SAE'] == Ruta_SAE)&
        (turno_ccz['Franja'] == Franja)&
        (turno_ccz['Tipo Dia'] == Tipo_Dia)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_ccz.loc[filtro].empty:
        # Obtener el primer valor
        turnos = turno_ccz.loc[filtro, 'Turno'].iloc[0]
        return turnos if not pd.isna(turnos) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Turno_ccz'] = iri.apply(
    lambda row: calcular_turno(
        row['Linea SAE'],
        row['Ruta SAE'],
        row['franja'],
        row['Tipo_dia']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via,Turno_ccz
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,NaN,1,NaN,NaN,<NA>,None,Festivo,F,None,None
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,NaN,1,16:02,1/03/2026,16,16-14,Festivo,F,PLA2,ET4T
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,NaN,1,5:00,1/03/2026,5,None,Festivo,F,None,None
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,NaN,1,5:44,1/03/2026,5,None,Festivo,F,None,None
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,NaN,1,6:28,1/03/2026,6,None,Festivo,F,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,NaN,1,17:14,31/03/2026,17,1-sep,Hábil,H,None,None
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,NaN,1,17:55,31/03/2026,17,1-sep,Hábil,H,None,None
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,NaN,1,18:38,31/03/2026,18,1-sep,Hábil,H,None,None
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,NaN,1,19:22,31/03/2026,19,1-sep,Hábil,H,None,None


In [15]:
#Traer el turno de supervisor

def calcular_turno(Linea,Ruta_SAE,Franja,Tipo_Dia):
    
    filtro = (
        (turno_sup['Linea'] == Linea)&
        (turno_sup['Ruta_SAE'] == Ruta_SAE)&
        (turno_sup['Franja'] == Franja)&
        (turno_sup['Tipo Dia'] == Tipo_Dia)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_sup.loc[filtro].empty:
        # Obtener el primer valor
        turno = turno_sup.loc[filtro, 'Estacion'].iloc[0]
        return turno if not pd.isna(turno) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Turno_Sup'] = iri.apply(
    lambda row: calcular_turno(
        row['Linea SAE'],
        row['Ruta SAE'],
        row['franja'],
        row['Tipo_dia']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via,Turno_ccz,Turno_Sup
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,1,NaN,NaN,<NA>,None,Festivo,F,None,None,None
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,1,16:02,1/03/2026,16,16-14,Festivo,F,PLA2,ET4T,SUP CCZ
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,1,5:00,1/03/2026,5,None,Festivo,F,None,None,None
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,1,5:44,1/03/2026,5,None,Festivo,F,None,None,None
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,1,6:28,1/03/2026,6,None,Festivo,F,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,1,17:14,31/03/2026,17,1-sep,Hábil,H,None,None,None
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,1,17:55,31/03/2026,17,1-sep,Hábil,H,None,None,None
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,1,18:38,31/03/2026,18,1-sep,Hábil,H,None,None,None
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,1,19:22,31/03/2026,19,1-sep,Hábil,H,None,None,None


In [16]:
#Traer el regulador según el turno

def calcular_turno(Fecha,TURNO_Programado):
    
    filtro = (
        (asistencia['Fecha'] == Fecha)&
        (asistencia['TURNO Programado'] == TURNO_Programado)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not asistencia.loc[filtro].empty:
        # Obtener el primer valor
        asistencias = asistencia.loc[filtro, 'Nombre Completo'].iloc[0]
        return asistencias if not pd.isna(asistencias) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['reg_via'] = iri.apply(
    lambda row: calcular_turno(
        row['Fecha Viaje'],
        row['Turno_via']
    ),
    axis=1
)

iri['reg_ccz'] = iri.apply(
    lambda row: calcular_turno(
        row['Fecha Viaje'],
        row['Turno_ccz']
    ),
    axis=1
)

iri['Supervisor'] = iri.apply(
    lambda row: calcular_turno(
        row['Fecha Viaje'],
        row['Turno_Sup']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via,Turno_ccz,Turno_Sup,reg_via,reg_ccz,Supervisor
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,<NA>,None,Festivo,F,None,None,None,None,None,None
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,16,16-14,Festivo,F,PLA2,ET4T,SUP CCZ,LORENA BENILDA PAJARO AREVALO,KAREN LORENA BARRERA SERRANO,CAMILO ANDRES LOPEZ PENAGOS
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,5,None,Festivo,F,None,None,None,None,None,None
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,5,None,Festivo,F,None,None,None,None,None,None
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,6,None,Festivo,F,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,17,1-sep,Hábil,H,None,None,None,None,None,None
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,17,1-sep,Hábil,H,None,None,None,None,None,None
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,18,1-sep,Hábil,H,None,None,None,None,None,None
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,19,1-sep,Hábil,H,None,None,None,None,None,None


In [17]:
#Llevar identificación al regulador

def calcular_posicion(Nombre_completo):
    
    filtro = (
        (personal['Nombre Completo'] == Nombre_completo)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not personal.loc[filtro].empty:
        # Obtener el primer valor
        personas = personal.loc[filtro, 'Identificación'].iloc[0]
        return  personas  if not pd.isna( personas ) else None  
    
    return 0  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Id_via'] = iri.apply(
    lambda row: calcular_posicion(
        row['reg_via']
    ),
    axis=1
)

iri['Id_ccz'] = iri.apply(
    lambda row: calcular_posicion(
        row['reg_ccz']
    ),
    axis=1
)

iri['Id_sup'] = iri.apply(
    lambda row: calcular_posicion(
        row['Supervisor']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Tipo_dia,Turno_via,Turno_ccz,Turno_Sup,reg_via,reg_ccz,Supervisor,Id_via,Id_ccz,Id_sup
0,1/03/2026,URBANO,AD0691017,1,1,17,105,10691,12746,504224,...,F,None,None,None,None,None,None,0,0,0
1,1/03/2026,ALIMENTACION,CE0A60012,28,27,2,105,10495,10971,0,...,F,PLA2,ET4T,SUP CCZ,LORENA BENILDA PAJARO AREVALO,KAREN LORENA BARRERA SERRANO,CAMILO ANDRES LOPEZ PENAGOS,57290332,1031150803,1023870284
2,1/03/2026,URBANO,CE0650001,2,1,1,105,10266,10533,504391,...,F,None,None,None,None,None,None,0,0,0
3,1/03/2026,URBANO,CE0650001,3,2,1,105,10266,10533,504391,...,F,None,None,None,None,None,None,0,0,0
4,1/03/2026,URBANO,CE0650001,4,3,1,105,10266,10533,504391,...,F,None,None,None,None,None,None,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113752,31/03/2026,ALIMENTACION,CO11B0012,21,20,6,105,10527,12440,225,...,H,None,None,None,None,None,None,0,0,0
113753,31/03/2026,ALIMENTACION,CO11B0012,22,21,6,105,10527,12440,225,...,H,None,None,None,None,None,None,0,0,0
113754,31/03/2026,ALIMENTACION,CO11B0012,23,22,6,105,10527,12440,225,...,H,None,None,None,None,None,None,0,0,0
113755,31/03/2026,ALIMENTACION,CO11B0012,24,23,6,105,10527,12440,225,...,H,None,None,None,None,None,None,0,0,0


In [18]:
#Exportar archivo

iri.to_csv(f"Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/regularidad_{mes}.csv", index=False)